# Contrastive video textures — Colab demo

This notebook clones the project, installs dependencies with **uv**, downloads the checkpoints the code expects on fixed paths (SlowFast, VGGish, **SuperSloMo**), and runs a sample **training** command.

For synthesis / evaluation (`--evaluate`), you must train or resume checkpoints first; `--SF` only controls frame interpolation at evaluation time (see `contrastive_video_textures/main.py`), not SlowFast pretraining.

In [ ]:
!git clone --quiet https://github.com/KoniHD/Berkeley-CS289A-Final-Project.git
%cd Berkeley-CS289A-Final-Project

*Note:* This part takes a while due to a lot of old legacy dependencies

In [ ]:
# Install this package and its dependencies from pyproject.toml (not a requirements file).
!uv pip install --system -r pyproject.toml

# Download pre-trained weights

## Download pre-trained ResNet, Slowfast, VGGish

*Note:* This part takes a while

In [ ]:
import os
import torch
import shutil
from pathlib import Path

# 1. Mock the hardcoded directories for R3D and SlowFast
PRETRAINED = "/home/medhini/audio_video_gan/contrastive_video_textures/pretrained"
SF_CFG = "/home/medhini/audio_video_gan/contrastive_video_textures/slowfast_configs"
os.makedirs(PRETRAINED, exist_ok=True)
os.makedirs(SF_CFG, exist_ok=True)

# 2. Fast wgets for SlowFast into the hardcoded paths
print("1/3: Pulling SlowFast config & weights directly...")
!wget -qO {SF_CFG}/SLOWFAST_8X8_R50.yaml https://raw.githubusercontent.com/facebookresearch/SlowFast/master/configs/Kinetics/SLOWFAST_8x8_R50.yaml
!wget -qO {PRETRAINED}/SLOWFAST_8x8_R50.pkl https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/kinetics400/SLOWFAST_8x8_R50.pkl

# 3. VGGish Surgery - SAVED LOCALLY THIS TIME
print("2/3: Pulling VGGish and running key surgery...")
!wget -qO vgg_raw.pth https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth
fixed_vgg = {k.replace("embeddings", "fc") if k.startswith("embeddings") else k: v for k, v in torch.load("vgg_raw.pth", map_location="cpu").items()}
# THE FIX: Save it locally exactly where main.py line 338 expects it!
torch.save(fixed_vgg, "pytorch_vggish.pth")

# 4. The R3D-18 Reality Check
print("3/3: Fetching custom 1039-class R3D-18 weights via gdown...")
!pip install -q gdown
!mkdir -p /tmp/r3d_pretrained
!gdown --folder https://drive.google.com/drive/folders/1xbYbZ7rpyjftI_KCk6YuL-XrfQDz7Yd4 -O /tmp/r3d_pretrained --remaining-ok

# Forcefully yank the file out of the GDrive folder structure into Medhini's path
print("Yanking r3d18_KM_200ep.pth into the hardcoded path...")
r3d_candidates = list(Path("/tmp/r3d_pretrained").rglob("r3d18_KM_200ep.pth"))
shutil.copy(r3d_candidates[0], f"{PRETRAINED}/r3d18_KM_200ep.pth")

print("Dirty prep complete! VGGish is local, R3D is hardcoded. You are good to go.")

## Download SuperSloMo weights

In [ ]:
# Run from repo root (after %cd Berkeley-CS289A-Final-Project)
import os, subprocess

ckpt_dir = "contrastive_video_textures/ckpt"
os.makedirs(ckpt_dir, exist_ok=True)
out = os.path.join(ckpt_dir, "SuperSloMo.ckpt")

subprocess.run(["pip", "install", "-q", "gdown"], check=True)
subprocess.run([
    "gdown", "1IvobLDbRiBgZr3ryCRrWL8xDbMZ-KnpF", "-O", out
], check=True)

import torch
d = torch.load(out, map_location="cpu")
assert "state_dictAT" in d and "state_dictFC" in d, f"Unexpected keys: {d.keys()}"
print("OK:", out)

# Train encoder

In [ ]:
%cd contrastive_video_textures

In [ ]:
!python main.py \
    --vdata ../data \
    --model_type 1 \
    --window 20 \
    --stride 4 \
    --temp 0.1 \
    -th 0.0 \
    --batch_size 8 \
    -negs 14 \
    --video_list Clown-Fish \
    --enc_arch resnet18 \
    --lr 1e-4

# Generate video

In [ ]:
!python main.py \
    --vdata ../data \
    --model_type 1 \
    --window 20 \
    --stride 4 \
    --temp 0.1 \
    --threshold 0.3 \
    --batch_size 8 \
    --n_negs 14 \
    --video_list Clown-Fish \
    --enc_arch resnet18 \
    --evaluate \
    --overlay_png ../data/Clownfish.png